In [1]:
import re
import emoji
import pandas as pd


train_path = "data\DeepX_train.xlsx"
val_path = "data\DeepX_unlabeled.xlsx"
unlabeled_path = "data\DeepX_validation.xlsx"


train_df = pd.read_excel(train_path)
val_df = pd.read_excel(val_path)
unlabeled_df = pd.read_excel(unlabeled_path)
# Map common emojis to Arabic-compatible sentiment tokens
EMOJI_MAP = {
    "💙": "EMOJI_LOVE", "🔥": "EMOJI_FIRE", "😂": "EMOJI_LAUGH",
    "👍": "EMOJI_THUMBSUP", "😡": "EMOJI_ANGRY", "💔": "EMOJI_BROKENHEART",
    "😍": "EMOJI_HEART_EYES", "👎": "EMOJI_THUMBSDOWN",
}

def encode_emojis(text):
    for char, token in EMOJI_MAP.items():
        text = text.replace(char, f" {token} ")
    # Remove remaining emojis not in map
    text = emoji.replace_emoji(text, replace="")
    return text

def normalize_arabic(text):
    # Normalize alef variants
    text = re.sub(r"[أإآا]", "ا", text)
    # Normalize teh marbuta
    text = re.sub(r"ة", "ه", text)
    # Normalize yeh variants
    text = re.sub(r"[يى]", "ي", text)
    # Remove diacritics (tashkeel)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    # Remove tatweel (elongation character)
    text = re.sub(r"ـ", "", text)
    return text

def fix_elongation(text):
    # Collapse 3+ repeated chars → 2 (حلووووو → حلوو)
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

def split_fused_words(text):
    # Heuristic: insert space before known prefixes stuck to words
    prefixes = ["ال", "وال", "بال", "كال", "فال", "لل"]
    for p in prefixes:
        text = re.sub(rf"(\w)({p})(\w)", r"\1 \2\3", text)
    return text

def clean_text(text):
    text = str(text)
    text = encode_emojis(text)
    text = normalize_arabic(text)
    text = fix_elongation(text)
    text = split_fused_words(text)
    # Remove punctuation except Arabic ones
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [2]:
ASPECTS = ["food","service","price","cleanliness",
           "delivery","ambiance","app_experience","general","none"]
SENTIMENTS = ["positive","negative","neutral"]

# Build all valid combos → 25 labels + none_neutral = 26
LABELS = []
for a in ASPECTS:
    for s in SENTIMENTS:
        if a == "none" and s != "neutral":
            continue  # none only maps to neutral
        LABELS.append(f"{a}_{s}")

LABEL2IDX = {l: i for i, l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)  # 26

def encode_labels(aspect_sentiments: dict) -> list:
    vec = [0.0] * NUM_LABELS
    for aspect, sentiment in aspect_sentiments.items():
        key = f"{aspect}_{sentiment}"
        if key in LABEL2IDX:
            vec[LABEL2IDX[key]] = 1.0
    return vec

def decode_labels(logits, thresholds=None) -> list:
    import torch
    probs = torch.sigmoid(torch.tensor(logits))
    if thresholds is None:
        thresholds = [0.5] * NUM_LABELS
    results = []
    for i, (prob, thresh) in enumerate(zip(probs, thresholds)):
        if prob >= thresh:
            parts = LABELS[i].rsplit("_", 1)
            results.append({"aspect": parts[0], "sentiment": parts[1]})
    if not results:
        results.append({"aspect": "none", "sentiment": "neutral"})
    return results

In [3]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn

MODEL_NAME = "aubmindlab/bert-base-arabertv2"

class ABSADataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df["review_text"].apply(clean_text).tolist()
        self.labels = df["aspect_sentiments"].apply(encode_labels).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.float)
        }

class ABSAModel(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_labels=NUM_LABELS, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden = self.bert.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Use [CLS] token
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(cls)

ModuleNotFoundError: No module named 'transformers'

In [ ]:
import numpy as np
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

def compute_pos_weight(df):
    """Compute positive class weights per label for BCEWithLogitsLoss."""
    all_labels = np.array(df["aspect_sentiments"].apply(encode_labels).tolist())
    pos = all_labels.sum(axis=0) + 1e-6
    neg = len(all_labels) - pos + 1e-6
    return torch.tensor(neg / pos, dtype=torch.float)

def focal_loss(logits, targets, gamma=2.0, pos_weight=None):
    """Focal loss reduces weight of easy negatives — great for imbalanced ABSA."""
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=pos_weight, reduction="none"
    )
    probs = torch.sigmoid(logits)
    p_t = probs * targets + (1 - probs) * (1 - targets)
    focal_weight = (1 - p_t) ** gamma
    return (focal_weight * bce).mean()

def train_epoch(model, loader, optimizer, scheduler, pos_weight, device):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        loss = focal_loss(logits, labels, gamma=2.0,
                          pos_weight=pos_weight.to(device))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# ── Training setup ─────────────────────────────────────────────────────────
EPOCHS = 5
LR = 2e-5
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds  = ABSADataset(train_df, tokenizer)
val_ds    = ABSADataset(val_df, tokenizer)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=32)

model  = ABSAModel().to("cuda")
pos_wt = compute_pos_weight(train_df)

optimizer = AdamW(
    [{"params": model.bert.parameters(), "lr": LR},
     {"params": model.classifier.parameters(), "lr": LR * 10}],
    weight_decay=0.01
)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

def get_all_logits(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch["input_ids"].to(device),
                           batch["attention_mask"].to(device))
            all_logits.append(logits.cpu().numpy())
            all_labels.append(batch["labels"].numpy())
    return np.vstack(all_logits), np.vstack(all_labels)

def tune_thresholds(logits, labels):
    """Find per-label threshold maximizing micro F1."""
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    best_thresholds = []
    for i in range(logits.shape[1]):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.1, 0.9, 0.05):
            preds = (probs[:, i] >= t).astype(int)
            f1 = f1_score(labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds.append(best_t)
    return best_thresholds

def evaluate(logits, labels, thresholds):
    probs = 1 / (1 + np.exp(-logits))
    preds = np.array([
        (probs[:, i] >= thresholds[i]).astype(int)
        for i in range(len(thresholds))
    ]).T
    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    per_label = f1_score(labels, preds, average=None, zero_division=0)
    print(f"Micro F1: {micro_f1:.4f}  |  Macro F1: {macro_f1:.4f}")
    for label, score in zip(LABELS, per_label):
        print(f"  {label:<30} {score:.3f}")
    return micro_f1

# ── Full training loop with early stopping ─────────────────────────────────
best_f1, patience, counter = 0, 3, 0
val_logits, val_labels = get_all_logits(model, val_loader, "cuda")
thresholds = tune_thresholds(val_logits, val_labels)

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, pos_wt, "cuda")
    val_logits, val_labels = get_all_logits(model, val_loader, "cuda")
    thresholds = tune_thresholds(val_logits, val_labels)
    val_f1 = evaluate(val_logits, val_labels, thresholds)
    print(f"Epoch {epoch+1}  loss={train_loss:.4f}  val_f1={val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({"model": model.state_dict(), "thresholds": thresholds}, "best_absa.pt")
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping.")
            break

In [ ]:
def predict(texts: list[str], model, tokenizer, thresholds, device="cuda"):
    model.eval()
    clean = [clean_text(t) for t in texts]
    enc = tokenizer(clean, max_length=128, padding=True,
                    truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(enc["input_ids"].to(device),
                       enc["attention_mask"].to(device)).cpu().numpy()

    results = []
    for row_logits in logits:
        results.append(decode_labels(row_logits, thresholds))
    return results

# ── Load best model and run ────────────────────────────────────────────────
ckpt = torch.load("best_absa.pt")
model.load_state_dict(ckpt["model"])
thresholds = ckpt["thresholds"]

# Single review
output = predict(["الاكل حلوووو بس الخدمهسيئه"], model, tokenizer, thresholds)
print(output[0])
# [{"aspect": "food", "sentiment": "positive"},
#  {"aspect": "service", "sentiment": "negative"}]

In [ ]:
def pseudo_label(unlabeled_df, model, tokenizer, thresholds,
                 confidence_threshold=0.85, device="cuda"):
    """
    Assign labels to unlabeled samples where model is highly confident.
    High confidence = at least one label prob > confidence_threshold,
    and no label prob sits in the uncertain zone [0.35, 0.65].
    """
    texts = unlabeled_df["review_text"].apply(clean_text).tolist()
    enc = tokenizer(texts, max_length=128, padding=True,
                    truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(enc["input_ids"].to(device),
                       enc["attention_mask"].to(device)).cpu().numpy()

    probs = 1 / (1 + np.exp(-logits))
    pseudo_rows = []
    for i, (row_probs, row_logits) in enumerate(zip(probs, logits)):
        max_prob = row_probs.max()
        uncertain = ((row_probs > 0.35) & (row_probs < 0.65)).any()
        if max_prob >= confidence_threshold and not uncertain:
            aspect_sentiments = {}
            for j, (p, t) in enumerate(zip(row_probs, thresholds)):
                if p >= t:
                    parts = LABELS[j].rsplit("_", 1)
                    aspect_sentiments[parts[0]] = parts[1]
            if not aspect_sentiments:
                aspect_sentiments = {"none": "neutral"}
            row = unlabeled_df.iloc[i].copy()
            row["aspect_sentiments"] = aspect_sentiments
            pseudo_rows.append(row)

    pseudo_df = pd.DataFrame(pseudo_rows)
    print(f"Pseudo-labeled {len(pseudo_df)}/{len(unlabeled_df)} samples")
    return pseudo_df

# Add to training data and retrain
pseudo_df = pseudo_label(unlabeled_df, model, tokenizer, thresholds)
augmented_train_df = pd.concat([train_df, pseudo_df]).reset_index(drop=True)